In [1]:
from dotenv import load_dotenv
load_dotenv()

True

---

#### 문서 로드

In [3]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/KCI_FI003153549_p5.pdf")
documents = loader.load()

#### 문서 분할

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

#### 임베딩 모델

In [6]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="bge-m3:latest",
)

#### 임베딩 & FAISS(Facebook AI Similarity Search) 벡터스토어 생성 및 저장

In [ ]:
# In-memory

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splitted_documents, embedding_model)

In [ ]:
# 로컬 저장
vectorstore.save_local("./faiss_index")

In [10]:
vectorstore

In [8]:
vectorstore = None

In [9]:
vectorstore

In [10]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    cached_embedder,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [11]:
vectorstore

---

In [11]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

In [12]:
results = vectorstore.similarity_search(query, k=5) # 검색을 외부에서 미리 실행한 후 반환된 결과 사용

In [18]:
results

[Document(id='04c840cd-1a10-4fad-b80a-835c6c9f7976', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator': 'PDFium', 'Producer': 'PDFium'}, page_content='1.2 Validity of Collected Data\n본 연구에서는 의료기기 임상시험에 특화된 Private\n수집된 데이터셋은 의료기기 임상시험에 특화된\nLLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화\nPrivate LLM 구축을 위해 도메인 적합성과 다양성, 그리\n데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적\n고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총\n용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로\n111,954페이지로 구성된 데이터는 의료기기 임상시험의\n구성된다. 각 단계는 의료기기 임상시험 분야의 특수성을\n규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포\n반영하여 상호 유기적으로 작동하며, Figure 1와 같이 이\n괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수\n를 통해 해당 분야에서 최적의 성능을 달성하도록 설계되\n있는 다양한 시나리오를 반영하도록 설계되었다.\n었다.'),
 Document(id='01de403b-a7c1-402e-9f0a-97096c010f83', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator

In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    '''다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트:{context}

질문: {question}
'''
)

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 컨텍스트만 사용해 질문에 답하세요.\n컨텍스트:{context}\n\n질문: {question}\n')

In [15]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

llm = OllamaLLM(model="gemma3:270m", base_url="http://localhost:11434") # 도커를 이용하고 있으므로 base_url을 지정해주어야 함

chain = prompt | llm | StrOutputParser()

In [16]:
response = chain.invoke({'context': results, 'question': query})

In [17]:
print(response)

본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 다음과 같습니다.

*   **문서 유형별 비율:**
    *   문서 유형 1: 111,954페이지
    *   문서 유형 2: 29149856-a517-4c9a-a639-878bb6bd5678
    *   문서 유형 3: 6e1bb174-3163-410a-8ebf-65b86d8fec79
    *   문서 유형 4: 29149856-a517-4c9a-a639-878bb6bd5678
    *   문서 유형 5: 7e1bb174-3163-4237-92f1-ef19ce953042

따라서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 **111,954페이지**입니다.


---

In [25]:
# 리트리버 생성
retriever = vectorstore.as_retriever()

In [26]:
retriever.invoke(query)

[Document(id='97a0cb49-1d4e-4e24-98b0-f56a1dbfe4fb', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator': 'PDFium', 'Producer': 'PDFium'}, page_content='1.2 Validity of Collected Data\n본 연구에서는 의료기기 임상시험에 특화된 Private\n수집된 데이터셋은 의료기기 임상시험에 특화된\nLLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화\nPrivate LLM 구축을 위해 도메인 적합성과 다양성, 그리\n데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적\n고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총\n용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로\n111,954페이지로 구성된 데이터는 의료기기 임상시험의\n구성된다. 각 단계는 의료기기 임상시험 분야의 특수성을\n규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포\n반영하여 상호 유기적으로 작동하며, Figure 1와 같이 이\n괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수\n를 통해 해당 분야에서 최적의 성능을 달성하도록 설계되\n있는 다양한 시나리오를 반영하도록 설계되었다.\n었다.'),
 Document(id='61aa3b8b-4257-40ef-b7e9-75faaa14c9fc', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama.llms import OllamaLLM

llm = OllamaLLM(model="gemma3:270m", base_url="http://localhost:11434") # 도커를 이용하고 있으므로 base_url을 지정해주어야 함

# retriever, RunnablePassthrough 객체 전달
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | llm
    | StrOutputParser()
)

In [28]:
response = chain.invoke(query) # query는 RunnablePassthrough()를 통과하여 question이라는 키의 값이 됨

In [29]:
print(response)

본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수는 **11,954 페이지**이며, 총 158개의 문서로 구성되어 있습니다.

문서 유형별 비율은 다음과 같습니다:
*   규제 문서: 30%
*   교육 자료: 20%
*   프로토콜 및 보고서: 25%
*   의료기기 특화 문서: 15%
*   기타: 10%
